# 05_step3_ERA5_downscaling_F2 - ERA5 1 deg -> 0.25 deg downscaling (global): SKELETON   [PLAN A]

**The downscaling method is NOT written here**: it will be provided with your supervisor.
This notebook only prepares the data and calls two functions to be filled in `f2_downscaler.py`.

**Principle (reminder)**: F2 is learned on ERA5 only (synchronised pairs ERA5 1 deg -> ERA5 0.25 deg, same day, same place),
then applied to the "ERA5-like 1 deg" GCM (notebook step4). The learning does not depend on any GCM dataset.

What the notebook prepares:
- `coarse`: ERA5 regridded to 1 deg (output of notebook 02), whole globe;
- `fine`: raw ERA5 at 0.25 deg, whole globe, same days;
- a **validation** mode that excludes 2015-2025 from the training (to test F2 on those years, notebook 09).

Volume: global ERA5 at 0.25 deg is ~1.5 GB per variable and per year (float32, uncompressed): the method will need to process the data in blocks (years, or spatial tiles).

In [ ]:
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

TRAIN_PERIOD = ("1981", "2025")
VALIDATION_MODE = False                  # True: train WITHOUT 2015-2025 (for the validation test)
EXCLUDE_PERIOD = ("2015", "2025")
REGRID_DIR = bu.out_dir(cfg, "plan_A", "regrid_1deg")
F2_DIR = bu.out_dir(cfg, "plan_A", "F2", "validation_without_2015_2025" if VALIDATION_MODE else "full")
import f2_downscaler as f2                # <- file to complete (supervisor's method)
print("F2 folder:", F2_DIR)

## 1. Preparation of the ERA5 pairs (1 deg -> 0.25 deg)

In [ ]:
def drop_period(da, period):
    mask = ~((da.time.dt.year >= int(period[0])) & (da.time.dt.year <= int(period[1])))
    return da.isel(time=np.where(mask.values)[0])

coarse, fine = {}, {}
for v in VARS:
    c = bu.read_processed(REGRID_DIR / f"era5_{v}_1deg.nc", v)
    f = bu.read_raw(cfg, "era5", v)
    c, f = bu.subset_time(c, *TRAIN_PERIOD), bu.subset_time(f, *TRAIN_PERIOD)
    common = np.intersect1d(c.time.values, f.time.values)
    c, f = c.sel(time=common), f.sel(time=common)
    if VALIDATION_MODE:
        c, f = drop_period(c, EXCLUDE_PERIOD), drop_period(f, EXCLUDE_PERIOD)
    coarse[v], fine[v] = c, f
    print(f"{v:7s} days: {c.sizes['time']} | 1deg: {c.sizes['lat']}x{c.sizes['lon']} | 0.25deg: {f.sizes['lat']}x{f.sizes['lon']}")

## 2. Check that the grids are nested

Each 1 deg cell should contain ~4 x 4 cells of 0.25 deg (same extent).

In [ ]:
c, f = coarse["pr"], fine["pr"]
print("1deg extent   :", float(c.lat.min()), float(c.lat.max()), float(c.lon.min()), float(c.lon.max()))
print("0.25deg extent:", float(f.lat.min()), float(f.lat.max()), float(f.lon.min()), float(f.lon.max()))
print("Resolution ratio (lat, lon):", round(float(np.diff(c.lat).mean() / np.diff(f.lat).mean()), 2),
      round(float(np.diff(c.lon).mean() / np.diff(f.lon).mean()), 2))

## 3. Training (supervisor's method: to be written in `f2_downscaler.train_f2`)

In [ ]:
try:
    f2.train_f2(coarse, fine, F2_DIR)
    print("F2 trained ->", F2_DIR)
except NotImplementedError as e:
    print("PLACEHOLDER:", e)

## 4. Notes for the next steps

- Contract expected by notebook step4: `f2_downscaler.apply_f2(coarse_dict, F2_DIR, target_lat, target_lon) -> dict` of 0.25 deg DataArrays (see `f2_downscaler.py`).
- Variables to process: `pr`, `tasmax`, `tasmin` (check `tasmin < tasmax` after downscaling).
- For the validation test of the downscaling alone: rerun with `VALIDATION_MODE = True`, then apply to ERA5 1 deg over 2015-2025 and compare with ERA5 0.25 deg (notebook 09).